
# 06 - Prespecified sensitivity analyses

Four prespecified checks on the patient-level analysis in `04_patient_level_inference.ipynb`.
None of them is a second discovery analysis: nothing here changes a tested family, a BH
denominator, or any reported q-value, and no check is described as validating another.

| # | Check | What it can and cannot say |
|---|---|---|
| 1 | Pooled downsampling stability across seeds 0-4 | measures how sensitive the **exploratory pooled-cell** analysis is to which cells were sampled. It says nothing about patient-level validity. |
| 2 | Leave-one-patient-out influence | measures how much a single patient moves each effect size. BH is **not** recomputed. |
| 3 | Clinical/technical balance | describes imbalance between outcome groups in the covariates that actually exist. |
| 4 | Censoring / survival structure | establishes what the outcome definition does and does not encode. |

A fifth arm - rebuilding endpoints at a 20-cell cell-type abundance threshold - is **not funded
for this run**. It is recorded as available future work and is not built here.

## Reporting language

Permitted: "associated with clinical-outcome group", "patient-level comparison", "exploratory
pooled-cell signal", "rank-biserial correlation" with confidence interval, "Benjamini-Hochberg
adjusted q-value". Not permitted: "caused", "protective", "drives outcome", "validated", or
"trend" as a substitute for a non-significant result.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:

# =====================================================================
# CONFIGURATION CELL - the single source of run parameters
# =====================================================================
from pathlib import Path
import sys

SCRIPT_DIR = Path(
    f"{ACC_DATA_ROOT}/scripts/named_scripts/"
    "cellphoneDB_scripts/clinical_outcome_v2"
)
OUTPUT_ROOT = Path(
    f"{ACC_DATA_ROOT}/outputs/cellphoneDB/clinical_outcome_v2"
)
CLINICAL_INFO_PATH = Path(f"{ACC_DATA_ROOT}/clinical_info.csv")

CONFIG_PATH = SCRIPT_DIR / "config.yaml"
PRIMARY_DIR = OUTPUT_ROOT / "primary"
SENSITIVITY_DIR = OUTPUT_ROOT / "sensitivity"
POOLED_DIR = OUTPUT_ROOT / "exploratory_pooled"
INPUTS_DIR = OUTPUT_ROOT / "inputs"
LOG_DIR = OUTPUT_ROOT / "logs"
LOG_PATH = LOG_DIR / "06_sensitivity_analysis.jsonl"
NOTEBOOK_ID = "06_sensitivity_analysis"

assert CONFIG_PATH.is_file(), f"config.yaml not found at {CONFIG_PATH}"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from utils import (  # noqa: E402  shared API - imported, never reimplemented
    load_config, log_jsonl, mannwhitney_hybrid, rank_biserial, bootstrap_ci,
    permutation_p, bh, fisher_availability,
)

CFG = load_config(CONFIG_PATH)


def cfg_get(key, default=None, required=True):
    if hasattr(CFG, "get") and not isinstance(CFG, (str, bytes)):
        try:
            if key in CFG:
                return CFG[key]
        except TypeError:
            pass
    if hasattr(CFG, key):
        return getattr(CFG, key)
    assert not required, f"required configuration key missing: {key}"
    return default


MIN_EVAL_PER_GROUP = int(cfg_get("min_evaluable_per_group"))
MIN_NONZERO_TOTAL = int(cfg_get("min_nonzero_total"))
FDR_ALPHA = float(cfg_get("fdr_alpha"))
ANALYSIS_SEED = int(cfg_get("analysis_seed"))
OUTCOME_COLUMN = str(cfg_get("outcome_column", "clinical_outcome", required=False))
POOLED_SEEDS = [int(s) for s in cfg_get("pooled_downsample_seeds", [0, 1, 2, 3, 4],
                                        required=False)]
SAMPLE_COLUMN = str(cfg_get("sample_column", "batch", required=False))
CELLTYPE_COLUMN = str(cfg_get("cell_type_column", "annot_6", required=False))
INPUT_H5AD = cfg_get("input_h5ad", None, required=False)
PATIENT_CELLTYPE_MIN_CELLS = int(cfg_get("patient_celltype_min_cells", 10, required=False))

assert OUTCOME_COLUMN == "clinical_outcome", OUTCOME_COLUMN
assert MIN_EVAL_PER_GROUP == 6 and MIN_NONZERO_TOTAL == 6
assert ANALYSIS_SEED == 20260731
assert PATIENT_CELLTYPE_MIN_CELLS == 10, (
    "this run is the 10-cell arm; the 20-cell abundance arm is not funded and is not built here")

# --- sensitivity-specific constants ---------------------------------------
INFLUENCE_Q_GATE = 0.10
INFLUENCE_TOP_N = 100
POOLED_FDR_THRESHOLD = 0.05
BALANCE_EXACT_ENUMERATION_LIMIT = 5_000_000
BALANCE_MC_RESAMPLES = 200_000
CENSORING_EVENT_TOKENS = ("event", "censor", "status", "followup", "follow_up",
                          "os_", "pfs", "dfs", "time_to", "survival")
NOT_FUNDED_ARMS = {
    "cell_abundance_20": {
        "status": "not funded for this run",
        "description": ("rebuild per-patient endpoints at a 20-cell cell-type abundance "
                        "threshold and rerun the patient-level pipeline, then compare effect "
                        "signs and ranks for endpoints estimable under both thresholds"),
        "note": "available future work; the 10-cell analysis remains primary",
    }
}

FORBIDDEN_PATH_TOKENS = ("cpdb_results_", "t1_cpdb_results", "t2_cpdb_results",
                         "t3_cpdb_results", "cellphoneDB_copy_backup")
for d in (SENSITIVITY_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:

# =====================================================================
# Imports, guards and shared-API adapters (mirrors 04; no new statistics)
# =====================================================================
import datetime as dt
import hashlib
import json
import platform
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import scipy
from scipy.stats import rankdata, norm

from IPython.display import display

ENVIRONMENT = {"python": sys.version.split()[0], "platform": platform.platform(),
               "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__}
RUN_STARTED = dt.datetime.now().isoformat(timespec="seconds")
CONFIG_SHA256 = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
WRITTEN_FILES = {}
BACKENDS = {}
STATUS = {}


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def guarded_path(path):
    p = Path(path).resolve()
    assert OUTPUT_ROOT.resolve() in p.parents, f"refusing to write outside output root: {p}"
    for token in FORBIDDEN_PATH_TOKENS:
        assert token not in str(p), f"refusing to write into a protected location: {p}"
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def write_table(df, path, index=False):
    p = guarded_path(path)
    df.to_csv(p, index=index)
    WRITTEN_FILES[str(p)] = {"rows": int(len(df)), "sha256": _sha256(p)}
    return p


def _try_calls(fn, *argsets, **kwargs):
    errors = []
    for args in argsets:
        try:
            return fn(*args, **kwargs)
        except TypeError as exc:
            errors.append(f"{args!r}: {exc}")
    raise TypeError(f"could not call {getattr(fn, '__name__', fn)}; tried: {errors}")


def _first_number(obj, names):
    for nm in names:
        if isinstance(obj, dict) and nm in obj:
            return obj[nm]
        if hasattr(obj, nm):
            return getattr(obj, nm)
    return None


def as_or_p(res):
    orr = _first_number(res, ("odds_ratio", "oddsratio", "or", "statistic"))
    p = _first_number(res, ("p", "pvalue", "p_value"))
    if orr is None or p is None:
        arr = list(res) if isinstance(res, (tuple, list)) else [res]
        orr, p = arr[0], arr[1]
    return float(orr), float(p)


def fisher_2x2(a_good, n_good, a_poor, n_poor):
    table = [[int(a_good), int(n_good - a_good)], [int(a_poor), int(n_poor - a_poor)]]
    try:
        res = _try_calls(fisher_availability,
                         (np.array([int(a_good), int(n_good)], dtype=int),
                          np.array([int(a_poor), int(n_poor)], dtype=int)),
                         (table,), (np.asarray(table),),
                         (table[0][0], table[0][1], table[1][0], table[1][1]))
        BACKENDS.setdefault("fisher", "utils.fisher_availability")
        return as_or_p(res)
    except (TypeError, AssertionError, ValueError) as exc:
        from scipy.stats import fisher_exact
        BACKENDS["fisher"] = f"scipy.fisher_exact ({exc})"
        r = fisher_exact(table)
        return float(r[0]), float(r[1])


def bh_qvalues(pvals):
    p = np.asarray(pvals, dtype=float)
    if p.size == 0:
        return p
    try:
        res = _try_calls(bh, (p,), (p, FDR_ALPHA))
        q = _first_number(res, ("q", "qvalues", "q_values", "pvals_corrected", "fdr"))
        if q is None:
            q = res[1] if isinstance(res, (tuple, list)) and len(res) > 1 else res
        q = np.asarray(q, dtype=float).ravel()
        assert q.size == p.size
        BACKENDS.setdefault("bh", "utils.bh")
    except (TypeError, AssertionError, ValueError, IndexError):
        from statsmodels.stats.multitest import multipletests
        q = multipletests(p, alpha=FDR_ALPHA, method="fdr_bh")[1]
        BACKENDS["bh"] = "statsmodels.multipletests"
    return np.clip(q, 0.0, 1.0)


def LOG(event, **fields):
    record = {"notebook": NOTEBOOK_ID, "event": event,
              "timestamp": dt.datetime.now().isoformat(timespec="seconds"),
              "config_sha256": CONFIG_SHA256, **fields}
    guarded_path(LOG_PATH)
    try:
        _try_calls(log_jsonl, (LOG_PATH, record), (record, LOG_PATH))
    except (TypeError, OSError):
        with open(LOG_PATH, "a") as fh:
            fh.write(json.dumps(record, default=str) + "\n")
    return record


def _parquet_engine():
    for mod in ("pyarrow", "fastparquet"):
        try:
            __import__(mod)
            return mod
        except ImportError:
            continue
    return None


PARQUET_ENGINE = _parquet_engine()
TABLE_SUFFIXES = [".parquet", ".csv", ".csv.gz", ".tsv", ".feather"]


def read_table_any(base, role):
    base = Path(base)
    stem = base.name[: -len(base.suffix)] if base.suffix else base.name
    tried = []
    for p in dict.fromkeys([base] + [base.with_name(stem + s) for s in TABLE_SUFFIXES]):
        tried.append(str(p))
        if not p.is_file():
            continue
        if p.suffix in (".parquet", ".feather"):
            if PARQUET_ENGINE is None:
                continue
            return (pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_feather(p)), p
        if p.suffix == ".tsv":
            return pd.read_csv(p, sep="\t"), p
        if p.name.endswith(".csv") or p.name.endswith(".csv.gz"):
            return pd.read_csv(p), p
    note = ("" if PARQUET_ENGINE else
            " No Parquet engine (pyarrow/fastparquet) is installed in this environment.")
    raise AssertionError(f"could not read the {role} table. Tried: {tried}.{note}")


def resolve_column(df, aliases, role, required=True):
    lower = {str(c).strip().lower(): c for c in df.columns}
    for a in aliases:
        if a.lower() in lower:
            return lower[a.lower()]
    assert not required, f"could not resolve {role}; tried {aliases}; have {list(df.columns)}"
    return None


LOG("start", environment=ENVIRONMENT, run_started=RUN_STARTED, parquet_engine=PARQUET_ENGINE)

In [ ]:

# =====================================================================
# Patient labels and the finalised patient-level results from 04
# =====================================================================
clinical = pd.read_csv(CLINICAL_INFO_PATH, encoding="utf-8-sig")
clinical.columns = [c.strip() for c in clinical.columns]
CLINICAL = clinical.rename(columns={"Sample ID": "patient",
                                    "Clinical outcome": "clinical_outcome",
                                    "Site of Extraction": "site_of_extraction",
                                    "Years": "years"})
CLINICAL["patient"] = CLINICAL["patient"].astype(str).str.strip()
for c in ("clinical_outcome", "site_of_extraction"):
    CLINICAL[c] = CLINICAL[c].astype(str).str.strip()
OUTCOME_MAP = dict(zip(CLINICAL["patient"], CLINICAL["clinical_outcome"]))

# clinical_info.csv contains ONLY these columns. Recorded so that the balance section
# cannot silently assume a covariate that does not exist.
CLINICAL_COLUMNS_PRESENT = list(clinical.columns)
CLINICAL_COVARIATES_ABSENT = ["age", "sex", "stage", "grade", "treatment", "tumour_purity",
                              "tumor_purity", "processing_batch", "sequencing_batch",
                              "event_indicator", "follow_up_time"]

RESULTS_PATH = PRIMARY_DIR / "patient_level_primary_results.csv"
SECONDARY_RESULTS_PATH = PRIMARY_DIR / "patient_level_secondary_results.csv"
assert RESULTS_PATH.is_file(), f"04_patient_level_inference has not been run: {RESULTS_PATH}"
PRIMARY_RESULTS = pd.read_csv(RESULTS_PATH)
SECONDARY_RESULTS = (pd.read_csv(SECONDARY_RESULTS_PATH)
                     if SECONDARY_RESULTS_PATH.is_file() else pd.DataFrame())
SUCCESS_04 = json.loads((PRIMARY_DIR / "_04_patient_level_inference_success.json").read_text())
assert SUCCESS_04["status"] == "success"
assert SUCCESS_04["outcome_column"] == "clinical_outcome"
PATIENTS = list(SUCCESS_04["patients"])
LABELS = np.array([OUTCOME_MAP[p] for p in PATIENTS])
GOOD_VEC = (LABELS == "Good").astype(float)
POOR_VEC = (LABELS == "Poor").astype(float)
N_GOOD_TOTAL, N_POOR_TOTAL = int(GOOD_VEC.sum()), int(POOR_VEC.sum())
assert (N_GOOD_TOTAL, N_POOR_TOTAL) == (SUCCESS_04["n_good"], SUCCESS_04["n_poor"])

display(pd.Series({"patients": len(PATIENTS), "Good": N_GOOD_TOTAL, "Poor": N_POOR_TOTAL,
                   "primary endpoints": len(PRIMARY_RESULTS),
                   "primary tested": int(PRIMARY_RESULTS["tested"].sum()),
                   "primary q<0.05": int(PRIMARY_RESULTS["significant_q05"].sum())},
                  name="inputs from 04"))


## 1. Pooled downsampling stability (seeds 0-4)

The exploratory pooled-cell analysis balances Good and Poor within each retained cell type by
downsampling, so its results depend on which cells the sampler happened to draw. Repeating it at
five seeds quantifies that dependence.

**What this section measures:** how reproducible the *exploratory pooled-cell* ranking is under
re-drawing cells. **What it does not measure:** whether any pooled signal holds at the patient
level. Cells within a patient are not independent replicates of the clinical outcome, so a
pooled result that is perfectly stable across seeds is still not patient-level evidence. Seed
stability and patient-level inference answer different questions, and a result appearing in both
sections is not thereby validated.

In [ ]:

# =====================================================================
# 1. Pooled downsampling stability
# =====================================================================
POOLED_FILE_PATTERNS = [
    "seed_{s}/wilcoxon_per_interaction_scores.csv",
    "seed{s}/wilcoxon_per_interaction_scores.csv",
    "downsample_seed_{s}/wilcoxon_per_interaction_scores.csv",
    "wilcoxon_per_interaction_scores_seed_{s}.csv",
    "wilcoxon_per_interaction_scores_seed{s}.csv",
    "seed_{s}_wilcoxon_per_interaction_scores.csv",
]
POOLED_GLOBS = ["**/*seed*{s}*/*per_interaction*.csv", "**/*seed*{s}*per_interaction*.csv"]

EFFECT_ALIASES = ["log2fc_good_vs_poor", "log2fc", "effect", "effect_size",
                  "median_diff_good_minus_poor", "rank_biserial"]
PVAL_ALIASES = ["pvalue", "p_value", "p"]
FDR_ALIASES = ["fdr", "q_value", "qvalue", "q", "padj"]
ID_ALIASES = ["id_cp_interaction", "interaction_id"]


def find_pooled_result(seed):
    for pat in POOLED_FILE_PATTERNS:
        p = POOLED_DIR / pat.format(s=seed)
        if p.is_file():
            return p
    for pat in POOLED_GLOBS:
        hits = sorted(POOLED_DIR.glob(pat.format(s=seed)))
        if len(hits) == 1:
            return hits[0]
        assert len(hits) <= 1, f"ambiguous pooled result files for seed {seed}: {hits}"
    return None


pooled_frames = {}
pooled_paths = {}
for s in POOLED_SEEDS:
    p = find_pooled_result(s)
    if p is not None:
        df = pd.read_csv(p)
        idc = resolve_column(df, ID_ALIASES, "id_cp_interaction")
        eff = resolve_column(df, EFFECT_ALIASES, "pooled effect")
        pv = resolve_column(df, PVAL_ALIASES, "pooled p-value")
        fdrc = resolve_column(df, FDR_ALIASES, "pooled fdr", required=False)
        sub = df[[idc, eff, pv] + ([fdrc] if fdrc else [])].rename(
            columns={idc: "id_cp_interaction", eff: "pooled_effect", pv: "pooled_p"})
        if fdrc:
            sub = sub.rename(columns={fdrc: "pooled_fdr"})
        else:
            sub["pooled_fdr"] = bh_qvalues(sub["pooled_p"].to_numpy())
        sub["seed"] = s
        sub["rank"] = sub["pooled_p"].rank(method="average")
        sub["selected"] = sub["pooled_fdr"] < POOLED_FDR_THRESHOLD
        pooled_frames[s] = sub
        pooled_paths[s] = str(p)

STATUS["pooled_seeds_found"] = sorted(pooled_frames)
STATUS["pooled_seeds_missing"] = sorted(set(POOLED_SEEDS) - set(pooled_frames))
assert len(pooled_frames) >= 2, (
    "downsampling stability needs at least two pooled seed runs; found "
    f"{sorted(pooled_frames)} under {POOLED_DIR}. Run 05_pooled_exploratory_analysis first.")
LOG("pooled_seeds_discovered", paths=pooled_paths, missing=STATUS["pooled_seeds_missing"])

pooled_all = pd.concat(pooled_frames.values(), ignore_index=True)
pooled_all["analysis_scope"] = "exploratory_pooled"
n_seeds = len(pooled_frames)

stability = (pooled_all.groupby("id_cp_interaction")
             .agg(n_seeds_present=("seed", "nunique"),
                  n_seeds_selected=("selected", "sum"),
                  effect_mean=("pooled_effect", "mean"),
                  effect_sd=("pooled_effect", "std"),
                  effect_min=("pooled_effect", "min"),
                  effect_max=("pooled_effect", "max"),
                  rank_mean=("rank", "mean"),
                  rank_sd=("rank", "std"),
                  rank_min=("rank", "min"),
                  rank_max=("rank", "max"))
             .reset_index())
stability["selection_frequency"] = stability["n_seeds_selected"] / n_seeds
stability["effect_range"] = stability["effect_max"] - stability["effect_min"]
stability["sign_stable"] = (
    pooled_all.assign(sgn=np.sign(pooled_all["pooled_effect"]))
    .groupby("id_cp_interaction")["sgn"].nunique().reindex(stability["id_cp_interaction"]).values
) == 1
stability["analysis_scope"] = "exploratory_pooled"
stability["interpretation"] = (
    "downsampling stability of an exploratory pooled-cell signal; not patient-level evidence")
write_table(stability.sort_values("selection_frequency", ascending=False),
            SENSITIVITY_DIR / "pooled_downsampling_stability.csv")

sel = {s: set(pooled_frames[s].loc[pooled_frames[s]["selected"], "id_cp_interaction"])
       for s in pooled_frames}
jac_rows = []
for a, b in combinations(sorted(sel), 2):
    inter, union = len(sel[a] & sel[b]), len(sel[a] | sel[b])
    jac_rows.append({"seed_a": a, "seed_b": b, "n_selected_a": len(sel[a]),
                     "n_selected_b": len(sel[b]), "n_intersection": inter, "n_union": union,
                     "jaccard": (inter / union) if union else np.nan})
JACCARD = pd.DataFrame(jac_rows)
write_table(JACCARD, SENSITIVITY_DIR / "pooled_downsampling_jaccard.csv")

POOLED_SUMMARY = pd.DataFrame([{
    "analysis_scope": "exploratory_pooled",
    "n_seeds_available": n_seeds,
    "seeds": ",".join(str(s) for s in sorted(pooled_frames)),
    "seeds_missing": ",".join(str(s) for s in STATUS["pooled_seeds_missing"]) or "none",
    "pooled_fdr_threshold": POOLED_FDR_THRESHOLD,
    "n_interactions_union": int(stability["id_cp_interaction"].nunique()),
    "n_selected_in_all_seeds": int((stability["n_seeds_selected"] == n_seeds).sum()),
    "n_selected_in_exactly_one_seed": int((stability["n_seeds_selected"] == 1).sum()),
    "n_selected_in_any_seed": int((stability["n_seeds_selected"] > 0).sum()),
    "median_jaccard": float(JACCARD["jaccard"].median()) if len(JACCARD) else np.nan,
    "min_jaccard": float(JACCARD["jaccard"].min()) if len(JACCARD) else np.nan,
    "median_effect_sd": float(stability["effect_sd"].median()),
    "median_rank_sd": float(stability["rank_sd"].median()),
    "fraction_sign_stable": float(stability["sign_stable"].mean()),
    "interpretation": ("downsampling sensitivity of the exploratory pooled-cell analysis; "
                       "NOT patient-level validity"),
}])
write_table(POOLED_SUMMARY, SENSITIVITY_DIR / "pooled_downsampling_summary.csv")
display(POOLED_SUMMARY.T)
display(JACCARD)
LOG("pooled_stability_complete", **POOLED_SUMMARY.iloc[0].to_dict())


## 2. Leave-one-patient-out influence

With 13 Good and 11 Poor patients, a single patient carries between 1/13 and 1/11 of a group.
This section removes one patient at a time and recomputes both effect sizes for the prespecified
influence subset - endpoints with BH `q < 0.10` plus the 100 smallest raw p-values in each family
- and reports sign flips and the maximum absolute change.

**Benjamini-Hochberg is not recomputed.** Re-running the correction inside each leave-one-out
replicate would manufacture apparent significance from a smaller, differently-composed family;
the q-values reported anywhere in this project remain the ones computed once in
`04_patient_level_inference.ipynb`.

**The concern this addresses.** In the earlier pooled-cell analysis, three Poor patients supplied
about 61% of all Poor cells, with P23 alone contributing about 25.5%. In a pooled test where
cells are the unit, an interaction that looks Poor-enriched can mechanically be an
P23-enriched interaction. The patient-level design removes that failure mode by construction
- every patient contributes exactly one value per endpoint - but a single patient can still be
influential through the *values* rather than through cell counts, and that is what is measured
here. Where per-patient cell counts are available they are tabulated alongside, so the pooled
imbalance is documented rather than asserted from memory.

In [ ]:

# =====================================================================
# 2a. Rebuild the patient x endpoint matrices used by 04
# =====================================================================
PATIENT_ALIASES = ["patient", "patient_id", "sample", "sample_id", "batch", "Sample ID"]
SENDER_ALIASES = ["sender", "cell_type_a", "source", "celltype_a", "cell_a"]
RECEIVER_ALIASES = ["receiver", "cell_type_b", "target", "celltype_b", "cell_b"]
SCORE_ALIASES = ["score", "interaction_score", "value", "mean_score", "cpdb_score"]
INTERACTION_ALIASES = ["id_cp_interaction", "id_cp_interactions", "interaction_id"]

scores_long, SCORES_PATH = read_table_any(
    PRIMARY_DIR / "patient_endpoint_scores_long.parquet", "patient endpoint scores")
S = {r: resolve_column(scores_long, a, r) for r, a in (
    ("patient", PATIENT_ALIASES), ("interaction", INTERACTION_ALIASES),
    ("sender", SENDER_ALIASES), ("receiver", RECEIVER_ALIASES), ("score", SCORE_ALIASES))}
scores_long[S["patient"]] = scores_long[S["patient"]].astype(str).str.strip()
scores_long = scores_long[scores_long[S["patient"]].isin(PATIENTS)]

wide = scores_long.pivot_table(
    index=[S["interaction"], S["sender"], S["receiver"]], columns=S["patient"],
    values=S["score"], aggfunc="first", dropna=False).reindex(columns=PATIENTS)
ENDPOINT_KEYS = wide.index.to_frame(index=False)
ENDPOINT_KEYS.columns = ["id_cp_interaction", "sender", "receiver"]
V = wide.to_numpy(dtype=float)
A = np.isfinite(V)
KEY_INDEX = {tuple(r): i for i, r in enumerate(ENDPOINT_KEYS.to_numpy())}
assert len(KEY_INDEX) == len(ENDPOINT_KEYS), "duplicate endpoint keys"
LOG("matrices_rebuilt", endpoints=int(V.shape[0]), patients=int(V.shape[1]),
    source=str(SCORES_PATH))


def rank_matrix_and_tie_terms(Vm, Am):
    filled = np.where(Am, Vm, np.inf)
    R = rankdata(filled, axis=1, method="average")
    order = np.argsort(filled, axis=1, kind="stable")
    Sm = np.take_along_axis(filled, order, axis=1)
    fin = np.isfinite(Sm)
    neq = np.ones(Sm.shape, dtype=bool)
    neq[:, 1:] = Sm[:, 1:] != Sm[:, :-1]
    gid = np.cumsum(neq, axis=1) - 1
    k, n = Sm.shape
    flat = np.repeat(np.arange(k), n) * n + gid.ravel()
    keep = fin.ravel()
    counts = np.bincount(flat[keep], minlength=k * n)
    t_elem = np.where(keep, counts[flat], 0).reshape(k, n).astype(float)
    return R, (t_elem ** 2).sum(axis=1) - fin.sum(axis=1)


def effects_for(Vm, Am, good_vec, poor_vec):
    R, _ = rank_matrix_and_tie_terms(Vm, Am)
    Af = Am.astype(float)
    n1, n2 = Af @ good_vec, Af @ poor_vec
    U1 = (R * Af) @ good_vec - n1 * (n1 + 1.0) / 2.0
    with np.errstate(invalid="ignore", divide="ignore"):
        rb = 2.0 * U1 / (n1 * n2) - 1.0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med_g = np.nanmedian(np.where(good_vec.astype(bool)[None, :], Vm, np.nan), axis=1)
        med_p = np.nanmedian(np.where(poor_vec.astype(bool)[None, :], Vm, np.nan), axis=1)
    return {"rank_biserial": rb, "median_difference": med_g - med_p,
            "n_good": n1.astype(int), "n_poor": n2.astype(int)}

In [ ]:

# =====================================================================
# 2b. Leave-one-patient-out recomputation (BH is NOT recomputed)
# =====================================================================
def influence_subset(results):
    t = results[results["tested"]].copy()
    if t.empty:
        return t
    keep = t["q_value"] < INFLUENCE_Q_GATE
    top = t.nsmallest(min(INFLUENCE_TOP_N, len(t)), "p_value").index
    keep.loc[top] = True
    return t[keep]


def leave_one_out(results, family):
    assert family == "primary", (
        "leave-one-out influence is defined on the directed primary endpoints; the secondary "
        "family is a fixed-pair-set summary of the same scores and is not re-derived here")
    sub = influence_subset(results)
    if sub.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    rows = np.array([KEY_INDEX[(r.id_cp_interaction, r.sender, r.receiver)]
                     for r in sub.itertuples()])
    Vs, As = V[rows], A[rows]
    full = effects_for(Vs, As, GOOD_VEC, POOR_VEC)
    assert np.allclose(full["rank_biserial"], sub["rank_biserial"].to_numpy(),
                       atol=1e-9, equal_nan=True), (
        "rebuilt effect sizes do not reproduce the 04 results; inputs have changed")

    recs = []
    for j, pat in enumerate(PATIENTS):
        keep = np.ones(len(PATIENTS), dtype=bool)
        keep[j] = False
        Aj = As & keep[None, :]
        e = effects_for(np.where(keep[None, :], Vs, np.nan), Aj, GOOD_VEC, POOR_VEC)
        still = (e["n_good"] >= MIN_EVAL_PER_GROUP) & (e["n_poor"] >= MIN_EVAL_PER_GROUP)
        recs.append(pd.DataFrame({
            "family": family,
            "id_cp_interaction": sub["id_cp_interaction"].to_numpy(),
            "sender": sub["sender"].to_numpy(),
            "receiver": sub["receiver"].to_numpy(),
            "left_out_patient": pat,
            "left_out_outcome": LABELS[j],
            "rank_biserial_full": full["rank_biserial"],
            "rank_biserial_loo": e["rank_biserial"],
            "delta_rank_biserial": e["rank_biserial"] - full["rank_biserial"],
            "median_difference_full": full["median_difference"],
            "median_difference_loo": e["median_difference"],
            "delta_median_difference": e["median_difference"] - full["median_difference"],
            "n_good_loo": e["n_good"], "n_poor_loo": e["n_poor"],
            "still_evaluable": still,
            "rank_biserial_sign_flip": (np.sign(e["rank_biserial"])
                                        * np.sign(full["rank_biserial"]) < 0),
            "median_difference_sign_flip": (np.sign(e["median_difference"])
                                            * np.sign(full["median_difference"]) < 0),
        }))
    loo = pd.concat(recs, ignore_index=True)

    per_endpoint = (loo.groupby(["family", "id_cp_interaction", "sender", "receiver"])
                    .agg(rank_biserial_full=("rank_biserial_full", "first"),
                         max_abs_delta_rank_biserial=("delta_rank_biserial",
                                                      lambda s: float(np.nanmax(np.abs(s)))),
                         max_abs_delta_median_difference=("delta_median_difference",
                                                          lambda s: float(np.nanmax(np.abs(s)))),
                         n_rank_biserial_sign_flips=("rank_biserial_sign_flip", "sum"),
                         n_median_difference_sign_flips=("median_difference_sign_flip", "sum"),
                         n_dropouts_unevaluable=("still_evaluable", lambda s: int((~s).sum())))
                    .reset_index())
    most = (loo.assign(absd=loo["delta_rank_biserial"].abs())
            .sort_values("absd", ascending=False)
            .drop_duplicates(["id_cp_interaction", "sender", "receiver"])
            [["id_cp_interaction", "sender", "receiver", "left_out_patient", "absd"]]
            .rename(columns={"left_out_patient": "most_influential_patient",
                             "absd": "max_abs_delta"}))
    per_endpoint = per_endpoint.merge(most, on=["id_cp_interaction", "sender", "receiver"],
                                      how="left")
    per_endpoint["bh_recomputed"] = False

    per_patient = (loo.groupby(["family", "left_out_patient", "left_out_outcome"])
                   .agg(n_endpoints=("delta_rank_biserial", "size"),
                        mean_abs_delta_rank_biserial=("delta_rank_biserial",
                                                      lambda s: float(np.nanmean(np.abs(s)))),
                        max_abs_delta_rank_biserial=("delta_rank_biserial",
                                                     lambda s: float(np.nanmax(np.abs(s)))),
                        n_rank_biserial_sign_flips=("rank_biserial_sign_flip", "sum"),
                        n_endpoints_made_unevaluable=("still_evaluable",
                                                      lambda s: int((~s).sum())))
                   .reset_index()
                   .sort_values("mean_abs_delta_rank_biserial", ascending=False))
    return loo, per_endpoint, per_patient


LOO, LOO_ENDPOINT, LOO_PATIENT = leave_one_out(PRIMARY_RESULTS, "primary")
if len(LOO):
    write_table(LOO, SENSITIVITY_DIR / "influence_leave_one_out_primary_long.csv")
    write_table(LOO_ENDPOINT, SENSITIVITY_DIR / "influence_per_endpoint_primary.csv")
    write_table(LOO_PATIENT, SENSITIVITY_DIR / "influence_per_patient_primary.csv")
    display(LOO_PATIENT)
    display(pd.Series({
        "endpoints_in_influence_subset": int(LOO_ENDPOINT.shape[0]),
        "endpoints_with_any_rank_biserial_sign_flip":
            int((LOO_ENDPOINT["n_rank_biserial_sign_flips"] > 0).sum()),
        "endpoints_with_any_median_difference_sign_flip":
            int((LOO_ENDPOINT["n_median_difference_sign_flips"] > 0).sum()),
        "max_abs_delta_rank_biserial_overall":
            float(LOO_ENDPOINT["max_abs_delta_rank_biserial"].max()),
        "median_max_abs_delta_rank_biserial":
            float(LOO_ENDPOINT["max_abs_delta_rank_biserial"].median()),
        "endpoints_losing_evaluability_for_some_dropout":
            int((LOO_ENDPOINT["n_dropouts_unevaluable"] > 0).sum()),
        "benjamini_hochberg_recomputed": False,
    }, name="leave-one-patient-out influence (primary)"))
STATUS["influence_subset_size"] = int(len(LOO_ENDPOINT))
LOG("influence_complete", subset=int(len(LOO_ENDPOINT)), bh_recomputed=False)

In [ ]:

# =====================================================================
# 2c. Per-patient cell contribution to the pooled analysis (documented, not assumed)
# =====================================================================
COUNTS_CANDIDATES = [INPUTS_DIR / "per_sample_celltype_counts.csv",
                     OUTPUT_ROOT / "cpdb_runs" / "per_sample_celltype_counts.csv",
                     PRIMARY_DIR / "per_sample_celltype_counts.csv"]
counts_path = next((p for p in COUNTS_CANDIDATES if p.is_file()), None)
if counts_path is None:
    STATUS["pooled_cell_contribution"] = (
        "per_sample_celltype_counts.csv not found; the pooled-pool shares quoted in the markdown "
        "above are carried from the earlier analysis and are not recomputed here")
    display(pd.Series({"status": STATUS["pooled_cell_contribution"]},
                      name="per-patient cell contribution"))
else:
    cnt = pd.read_csv(counts_path)
    pcol = resolve_column(cnt, PATIENT_ALIASES, "patient")
    ncol = next((c for c in cnt.columns if str(c).lower() in
                 {"n_cells", "cells", "count", "n"}), None)
    assert ncol is not None, f"no cell-count column in {counts_path}: {list(cnt.columns)}"
    per_pat = (cnt.groupby(pcol)[ncol].sum().rename("n_cells").reset_index()
               .rename(columns={pcol: "patient"}))
    per_pat["patient"] = per_pat["patient"].astype(str).str.strip()
    per_pat = per_pat[per_pat["patient"].isin(PATIENTS)]
    per_pat["clinical_outcome"] = per_pat["patient"].map(OUTCOME_MAP)
    per_pat["share_of_outcome_pool"] = per_pat.groupby("clinical_outcome")["n_cells"].transform(
        lambda s: s / s.sum())
    per_pat = per_pat.sort_values(["clinical_outcome", "share_of_outcome_pool"],
                                  ascending=[True, False])
    per_pat["cumulative_share_within_outcome"] = per_pat.groupby(
        "clinical_outcome")["share_of_outcome_pool"].cumsum()
    per_pat["note"] = ("share of the pooled-cell pool, which is why the pooled analysis is "
                       "exploratory only; the patient-level analysis gives every patient "
                       "weight 1 by construction")
    write_table(per_pat, SENSITIVITY_DIR / "pooled_cell_contribution_by_patient.csv")
    top3 = (per_pat[per_pat["clinical_outcome"] == "Poor"]
            .nlargest(3, "share_of_outcome_pool"))
    STATUS["pooled_cell_contribution"] = {
        "top3_poor_share": float(top3["share_of_outcome_pool"].sum()),
        "largest_poor_patient": str(top3["patient"].iloc[0]),
        "largest_poor_share": float(top3["share_of_outcome_pool"].iloc[0]),
    }
    display(per_pat)
    LOG("pooled_cell_contribution", **STATUS["pooled_cell_contribution"])


## 3. Clinical and technical balance

### What is actually available

`clinical_info.csv` contains exactly these columns:

```
Sample ID, Site of Extraction, Years, Clinical outcome, Model outcome,
Bulk RNA Datasets, Single Cell RNA Datasets, Spatial Datasets
```

There is **no age, no sex, no stage or grade, no treatment, no tumour purity and no processing or
sequencing batch**. That is stated here as a limitation of the available metadata, not worked
around: no proxy is invented, no covariate is imputed, and no substitute variable is presented as
if it were one of the missing ones. Any confounding by those variables is unmeasurable in this
dataset and remains unmeasurable after this section.

The balance check therefore runs on `Site of Extraction`, plus whatever patient-constant technical
columns exist in `adata.obs` and the per-patient cell counts.

### Method and its limits

With 24 patients, per-site cell counts are single digits. Fisher's exact test is used for the
small categorical tables (each site against all others, BH corrected across sites) together with
an exact conditional test of the full site-by-outcome table, and standardized differences are
reported descriptively. **No multivariable model is fitted**: with 24 patients and roughly ten
site categories, any adjusted model would be fitting more parameters than it has information for,
and its coefficients would be arithmetic rather than evidence.

### What is expected to be found

Site of extraction is strongly imbalanced by outcome: Maxilla 0 Good / 4 Poor, Parotid Gland
3 / 0, Larynx 2 / 0, Tongue 2 / 0, Unknown 4 Good / 1 Poor, Base of Tongue 2 / 2. Several sites
are perfectly separated by outcome. Two consequences follow and are both reported:

1. **Plausible confounding.** An endpoint associated with clinical-outcome group may be
   associated with anatomical site instead, and this design cannot separate the two. No result in
   this project should be read as outcome-specific rather than site-specific.
2. **Patient-label exchangeability may fail.** The permutation check in
   `04_patient_level_inference.ipynb` permutes outcome labels across all 24 patients as if they
   were exchangeable. If site drives part of the interaction profile, they are not, and the
   permutation null is then optimistic. That is a limitation of the permutation check, and it is
   one more reason it is reported as a sensitivity check rather than as a discovery filter.

In [ ]:

# =====================================================================
# 3. Clinical / technical balance
# =====================================================================
def standardized_difference(p1, p2):
    """Descriptive standardized difference between two proportions."""
    pooled = (p1 * (1 - p1) + p2 * (1 - p2)) / 2.0
    return (p1 - p2) / np.sqrt(pooled) if pooled > 0 else (0.0 if p1 == p2 else np.inf)


def exact_table_pvalue(table, seed=ANALYSIS_SEED, mc_resamples=BALANCE_MC_RESAMPLES):
    """Exact conditional p for an r x 2 table by enumeration, else a seeded Monte Carlo p."""
    from math import lgamma

    table = np.asarray(table, dtype=int)
    rows, cols, N = table.sum(1), table.sum(0), int(table.sum())

    def log_pmf(first_col):
        lg = sum(lgamma(r + 1) for r in rows) + sum(lgamma(c + 1) for c in cols) - lgamma(N + 1)
        for a, r in zip(first_col, rows):
            lg -= lgamma(a + 1) + lgamma(r - a + 1)
        return lg

    space = int(np.prod([r + 1 for r in rows]))
    obs = log_pmf(table[:, 0])
    if space <= BALANCE_EXACT_ENUMERATION_LIMIT:
        total = 0.0
        stack = [(0, 0, [])]
        while stack:
            i, used, acc = stack.pop()
            if i == len(rows):
                if used == cols[0]:
                    lp = log_pmf(acc)
                    if lp <= obs + 1e-12:
                        total += np.exp(lp)
                continue
            for a in range(0, min(rows[i], cols[0] - used) + 1):
                stack.append((i + 1, used + a, acc + [a]))
        return float(min(1.0, total)), "exact_enumeration", space
    rng = np.random.default_rng(seed)
    labels = np.repeat(np.arange(len(rows)), rows)
    hits = 0
    for _ in range(mc_resamples):
        perm = rng.permutation(N)
        first = np.bincount(labels[perm[: cols[0]]], minlength=len(rows))
        if log_pmf(first) <= obs + 1e-12:
            hits += 1
    return float((hits + 1) / (mc_resamples + 1)), "monte_carlo_permutation", space


COHORT = CLINICAL[CLINICAL["patient"].isin(PATIENTS)].copy()
assert len(COHORT) == len(PATIENTS)

site_tab = pd.crosstab(COHORT["site_of_extraction"], COHORT["clinical_outcome"])
for oc in ("Good", "Poor"):
    if oc not in site_tab.columns:
        site_tab[oc] = 0
site_tab = site_tab[["Good", "Poor"]].sort_index()
write_table(site_tab.reset_index(), SENSITIVITY_DIR / "balance_site_by_outcome.csv")

rows = []
for site, r in site_tab.iterrows():
    orr, p = fisher_2x2(int(r["Good"]), N_GOOD_TOTAL, int(r["Poor"]), N_POOR_TOTAL)
    rows.append({"covariate": "Site of Extraction", "level": site,
                 "n_good": int(r["Good"]), "n_poor": int(r["Poor"]),
                 "prop_good": r["Good"] / N_GOOD_TOTAL, "prop_poor": r["Poor"] / N_POOR_TOTAL,
                 "odds_ratio": orr, "fisher_p": p,
                 "standardized_difference": standardized_difference(
                     r["Good"] / N_GOOD_TOTAL, r["Poor"] / N_POOR_TOTAL),
                 "perfectly_separated": bool((r["Good"] == 0) != (r["Poor"] == 0))})
SITE_BALANCE = pd.DataFrame(rows)
SITE_BALANCE["fisher_q"] = bh_qvalues(SITE_BALANCE["fisher_p"].to_numpy())
SITE_BALANCE = SITE_BALANCE.sort_values("fisher_p")
write_table(SITE_BALANCE, SENSITIVITY_DIR / "balance_site_fisher.csv")

overall_p, overall_method, space = exact_table_pvalue(site_tab.to_numpy())
BALANCE_OVERALL = pd.DataFrame([{
    "covariate": "Site of Extraction",
    "n_levels": int(site_tab.shape[0]),
    "n_patients": int(site_tab.to_numpy().sum()),
    "test": "conditional exact test of independence, r x 2 table",
    "method": overall_method,
    "enumeration_space": int(space),
    "p_value": overall_p,
    "n_perfectly_separated_levels": int(SITE_BALANCE["perfectly_separated"].sum()),
    "max_abs_standardized_difference": float(
        SITE_BALANCE["standardized_difference"].replace([np.inf, -np.inf], np.nan).abs().max()),
    "multivariable_model_fitted": False,
    "interpretation": ("site of extraction is imbalanced by outcome; outcome-associated and "
                       "site-associated signals cannot be separated in this design, and patient-"
                       "label exchangeability may therefore fail"),
}])
write_table(BALANCE_OVERALL, SENSITIVITY_DIR / "balance_overall_test.csv")

MISSING_COVARIATES = pd.DataFrame({
    "covariate": CLINICAL_COVARIATES_ABSENT,
    "present_in_clinical_info": [c in [x.lower().replace(" ", "_")
                                       for x in CLINICAL_COLUMNS_PRESENT]
                                 for c in CLINICAL_COVARIATES_ABSENT],
    "handling": "not available in this dataset; stated as a limitation, no proxy substituted",
})
write_table(MISSING_COVARIATES, SENSITIVITY_DIR / "balance_absent_covariates.csv")

display(site_tab)
display(SITE_BALANCE)
display(BALANCE_OVERALL.T)
LOG("balance_complete", overall_p=overall_p, method=overall_method,
    n_perfectly_separated=int(SITE_BALANCE["perfectly_separated"].sum()))

In [ ]:

# =====================================================================
# 3b. Technical covariates from adata.obs, where any exist
# =====================================================================
TECH_ROWS = []
if INPUT_H5AD and Path(str(INPUT_H5AD)).is_file():
    try:
        import anndata as ad

        obs = ad.read_h5ad(str(INPUT_H5AD), backed="r").obs
        assert SAMPLE_COLUMN in obs.columns, (SAMPLE_COLUMN, list(obs.columns)[:30])
        obs = obs.copy()
        obs[SAMPLE_COLUMN] = obs[SAMPLE_COLUMN].astype(str).str.strip()
        obs = obs[obs[SAMPLE_COLUMN].isin(PATIENTS)]

        ncell = obs.groupby(SAMPLE_COLUMN).size().rename("n_cells").reset_index()
        ncell.columns = ["patient", "n_cells"]
        ncell["clinical_outcome"] = ncell["patient"].map(OUTCOME_MAP)
        g = ncell.loc[ncell["clinical_outcome"] == "Good", "n_cells"].to_numpy()
        p = ncell.loc[ncell["clinical_outcome"] == "Poor", "n_cells"].to_numpy()
        _u_cells, _, _ = mannwhitney_hybrid(g, p)
        rb_cells = float(rank_biserial(_u_cells, g.size, p.size))
        TECH_ROWS.append({"covariate": "n_cells_per_patient", "type": "numeric",
                          "median_good": float(np.median(g)), "median_poor": float(np.median(p)),
                          "rank_biserial_good_minus_poor": rb_cells, "p_value": np.nan,
                          "note": "technical covariate derived from adata.obs"})
        write_table(ncell, SENSITIVITY_DIR / "balance_cells_per_patient.csv")

        nunique = obs.groupby(SAMPLE_COLUMN).nunique(dropna=False)
        patient_constant = [c for c in nunique.columns
                            if c not in (SAMPLE_COLUMN, CELLTYPE_COLUMN)
                            and int(nunique[c].max()) == 1]
        for c in patient_constant:
            vals = obs.groupby(SAMPLE_COLUMN)[c].first()
            vals = vals.reindex(PATIENTS).astype(str)
            if vals.nunique() < 2 or vals.nunique() > 12:
                continue
            tab = pd.crosstab(vals, pd.Series(LABELS, index=PATIENTS))
            for oc in ("Good", "Poor"):
                if oc not in tab.columns:
                    tab[oc] = 0
            tab = tab[["Good", "Poor"]]
            pv, meth, _ = exact_table_pvalue(tab.to_numpy())
            for lvl, r in tab.iterrows():
                TECH_ROWS.append({
                    "covariate": c, "type": "categorical (patient-constant)", "level": lvl,
                    "n_good": int(r["Good"]), "n_poor": int(r["Poor"]),
                    "standardized_difference": standardized_difference(
                        r["Good"] / N_GOOD_TOTAL, r["Poor"] / N_POOR_TOTAL),
                    "table_p_value": pv, "table_method": meth,
                    "note": "technical covariate derived from adata.obs"})
        STATUS["technical_covariates"] = {
            "h5ad": str(INPUT_H5AD), "patient_constant_columns": patient_constant}
    except Exception as exc:  # noqa: BLE001 - recorded, never silently ignored
        STATUS["technical_covariates"] = f"unavailable: {type(exc).__name__}: {exc}"
else:
    STATUS["technical_covariates"] = (
        "input_h5ad not configured or not readable; no adata.obs technical covariates were "
        "assessed. This is a gap in the balance check, not evidence of balance.")

TECHNICAL_BALANCE = pd.DataFrame(TECH_ROWS)
if len(TECHNICAL_BALANCE):
    write_table(TECHNICAL_BALANCE, SENSITIVITY_DIR / "balance_technical_covariates.csv")
    display(TECHNICAL_BALANCE)
display(pd.Series({"technical covariate status": str(STATUS["technical_covariates"])[:400]},
                  name="adata.obs balance"))
LOG("technical_balance", status=STATUS["technical_covariates"])


## 4. Censoring and survival structure of the outcome definition

`Years` in `clinical_info.csv` takes only the values 2, 3 and 5, and `Clinical outcome` is a
deterministic function of it: 2 maps to Poor, and 3 and 5 map to Good. There is no event
indicator and no continuous follow-up time anywhere in the file.

Both consequences follow, and both are stated:

1. **Dichotomising loses nothing that this file records.** Because the outcome is already a
   deterministic recoding of a three-valued field, the Good/Poor split discards no information
   that the file actually contains. There is no hidden gradient being flattened.
2. **A survival-aware analysis cannot be built from this file.** Time-to-event analysis needs an
   event indicator and a follow-up time per patient; neither exists here. Whether `Years` is a
   follow-up duration, a survival time, or a binned category is not recorded, and a Cox or
   Kaplan-Meier analysis fitted to it would be inventing its own censoring structure.

No survival analysis is retrofitted. If event indicators and follow-up times are recoverable from
the source clinical records, a prespecified survival analysis would be a separate study with its
own protocol, not a reinterpretation of this one.

In [ ]:

# =====================================================================
# 4. Censoring check
# =====================================================================
years = COHORT["years"]
YEAR_VALUES = sorted(pd.unique(years.dropna()).tolist())
assert set(YEAR_VALUES) <= {2, 3, 5}, YEAR_VALUES

mapping = COHORT.groupby("years")["clinical_outcome"].nunique()
assert (mapping == 1).all(), (
    f"Years does not determine clinical outcome: {mapping.to_dict()}")
observed_map = COHORT.groupby("years")["clinical_outcome"].first().to_dict()
assert observed_map.get(2) == "Poor", observed_map
assert all(observed_map.get(y) == "Good" for y in (3, 5) if y in observed_map), observed_map

event_like = [c for c in CLINICAL_COLUMNS_PRESENT
              if any(tok in str(c).lower().replace(" ", "_") for tok in CENSORING_EVENT_TOKENS)]
assert not event_like, (
    f"an event/censoring-like column appeared in clinical_info.csv: {event_like}; "
    "the censoring statement in this notebook must be revisited")

CENSORING = pd.DataFrame([{
    "years_values_observed": ",".join(str(y) for y in YEAR_VALUES),
    "years_to_outcome_mapping": json.dumps({str(k): v for k, v in observed_map.items()}),
    "mapping_is_deterministic": True,
    "event_indicator_present": False,
    "continuous_follow_up_time_present": False,
    "dichotomisation_loses_recorded_information": False,
    "survival_analysis_constructible_from_this_file": False,
    "survival_analysis_retrofitted": False,
    "statement": ("Clinical outcome is a deterministic recoding of a three-valued Years field, "
                  "so the Good/Poor split discards no information present in this file; equally, "
                  "with no event indicator and no continuous follow-up time, a survival-aware "
                  "analysis cannot be constructed from it and none is attempted."),
}])
write_table(CENSORING, SENSITIVITY_DIR / "censoring_check.csv")
display(pd.crosstab(COHORT["years"], COHORT["clinical_outcome"]))
display(CENSORING.T)
LOG("censoring_check", years=YEAR_VALUES, mapping=observed_map, event_columns=event_like)


## 5. Arm not funded for this run

The 20-cell cell-type abundance sensitivity arm - rebuilding per-patient endpoints with a 20-cell
threshold, rerunning the patient-level pipeline, and comparing effect signs and ranks for
endpoints estimable under both thresholds - is **not funded for this run and is not built here**.
The 10-cell analysis remains primary. The cell below asserts that nothing in this run has quietly
produced or consumed a 20-cell artefact, so the record is unambiguous.

In [ ]:

# =====================================================================
# 5. Not-funded arm: recorded, asserted absent, not built
# =====================================================================
NOT_FUNDED = pd.DataFrame([{"arm": k, **v} for k, v in NOT_FUNDED_ARMS.items()])
write_table(NOT_FUNDED, SENSITIVITY_DIR / "arms_not_funded.csv")

leaked = [str(p) for p in OUTPUT_ROOT.rglob("*20*cell*") if p.is_file()]
leaked += [str(p) for p in OUTPUT_ROOT.rglob("*min20*") if p.is_file()]
assert not leaked, f"20-cell artefacts exist although the arm is not funded: {leaked}"
assert PATIENT_CELLTYPE_MIN_CELLS == 10
display(NOT_FUNDED.T)
LOG("not_funded_arms", arms=list(NOT_FUNDED_ARMS))

In [ ]:

# =====================================================================
# Verification and machine-readable success record
# =====================================================================
# 1. nothing here recomputed a q-value that 04 owns
assert "q_value" not in set(LOO.columns) if len(LOO) else True
if len(LOO_ENDPOINT):
    assert (~LOO_ENDPOINT["bh_recomputed"]).all(), "BH must not be recomputed per leave-one-out"
    assert len(LOO) == len(LOO_ENDPOINT) * len(PATIENTS), "leave-one-out grid is incomplete"
    assert set(LOO["left_out_patient"]) == set(PATIENTS)
    assert LOO["left_out_outcome"].isin(["Good", "Poor"]).all()

# 2. the pooled section is labelled exploratory everywhere it is written
assert (stability["analysis_scope"] == "exploratory_pooled").all()
assert POOLED_SUMMARY["interpretation"].str.contains("NOT patient-level validity").all()

# 3. balance section fitted no model and invented no covariate
assert not bool(BALANCE_OVERALL["multivariable_model_fitted"].iloc[0])
assert not MISSING_COVARIATES["present_in_clinical_info"].any(), (
    "a covariate assumed absent is actually present; the limitation text must be corrected")

# 4. censoring statement matches the data
assert not bool(CENSORING["survival_analysis_retrofitted"].iloc[0])
assert bool(CENSORING["mapping_is_deterministic"].iloc[0])

# 5. every written file is inside the sensitivity/logs area of the v2 output tree
for f in WRITTEN_FILES:
    assert str(SENSITIVITY_DIR) in f, f

SUCCESS = {
    "notebook": NOTEBOOK_ID,
    "status": "success",
    "run_started": RUN_STARTED,
    "run_finished": dt.datetime.now().isoformat(timespec="seconds"),
    "config_sha256": CONFIG_SHA256,
    "environment": ENVIRONMENT,
    "analysis_seed": ANALYSIS_SEED,
    "n_patients": len(PATIENTS), "n_good": N_GOOD_TOTAL, "n_poor": N_POOR_TOTAL,
    "checks": {
        "pooled_downsampling_stability": {
            "seeds_found": STATUS["pooled_seeds_found"],
            "seeds_missing": STATUS["pooled_seeds_missing"],
            "summary": POOLED_SUMMARY.iloc[0].to_dict(),
            "scope": "exploratory_pooled; not patient-level validity",
        },
        "influence_leave_one_out": {
            "subset_size": STATUS["influence_subset_size"],
            "bh_recomputed": False,
            "per_patient": (LOO_PATIENT.to_dict(orient="records") if len(LOO_PATIENT) else []),
            "pooled_cell_contribution": STATUS.get("pooled_cell_contribution"),
        },
        "clinical_technical_balance": {
            "overall": BALANCE_OVERALL.iloc[0].to_dict(),
            "by_site": SITE_BALANCE.to_dict(orient="records"),
            "technical_covariates": STATUS["technical_covariates"],
            "absent_covariates": CLINICAL_COVARIATES_ABSENT,
            "multivariable_model_fitted": False,
        },
        "censoring": CENSORING.iloc[0].to_dict(),
    },
    "arms_not_funded": NOT_FUNDED_ARMS,
    "statistical_backends": BACKENDS,
    "written_files": WRITTEN_FILES,
    "claims": {
        "pooled_seed_stability_is_patient_level_evidence": False,
        "leave_one_out_bh_recomputed": False,
        "survival_analysis_retrofitted": False,
        "multivariable_confounder_model_fitted": False,
        "twenty_cell_arm_built": False,
    },
}
SUCCESS_PATH = guarded_path(SENSITIVITY_DIR / "_06_sensitivity_analysis_success.json")
SUCCESS_PATH.write_text(json.dumps(SUCCESS, indent=2, default=str) + "\n")
LOG("success", success_path=str(SUCCESS_PATH))
assert SUCCESS_PATH.is_file()
display(pd.Series({"success record": str(SUCCESS_PATH),
                   "pooled seeds": len(STATUS["pooled_seeds_found"]),
                   "influence endpoints": STATUS["influence_subset_size"],
                   "site levels perfectly separated":
                       int(SITE_BALANCE["perfectly_separated"].sum()),
                   "files written": len(WRITTEN_FILES)},
                  name="06_sensitivity_analysis"))